In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks',
    '/home/export/soheuny/SRFinder/soheun/notebooks/draft'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("/home/export/soheuny/SRFinder/soheun")

In [2]:
from typing import Tuple, Any
from constants import FEATURES
from signal_region import compute_sr_stats, get_SR_CR_cut
from events_data import events_from_scdinfo
from dataset import MotherSamples
from training_info import TrainingInfo

SIGNAL_FILENAME = "HH4b_picoAOD.h5"

def load_events_data(hash_: str) -> Tuple[Any, Any]:
    """Load training and test events data from a hash."""
    CR_fvt_tinfo = TrainingInfo.load(hash_)
    SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
    smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
    msamples = MotherSamples.load(smeared_tinfo.ms_hash)
    
    events_train = events_from_scdinfo(
        msamples.scdinfo[smeared_tinfo.ms_idx], 
        FEATURES, 
        SIGNAL_FILENAME
    )
    events_tst = events_from_scdinfo(
        msamples.scdinfo[~smeared_tinfo.ms_idx], 
        FEATURES, 
        SIGNAL_FILENAME
    )
    return events_train, events_tst


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo
import pickle

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
# plt.rcParams["font.family"] = "serif"
# plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3
# legend font size
plt.rcParams["legend.fontsize"] = 15

import pandas as pd

path_3b = Path("../events/MG3/dataframes/threeTag_picoAOD.h5")
path_4b = Path("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
path_signal = Path("../events/MG3/dataframes/HH4b_picoAOD.h5")
df_3b = pd.read_hdf(path_3b)
df_bg4b = pd.read_hdf(path_4b)
df_signal = pd.read_hdf(path_signal)
df_3b["signal"] = False
df_bg4b["signal"] = False
df_signal["signal"] = True
raw_df_list = [df_3b, df_bg4b, df_signal]
loaded_df = {path_3b: df_3b, path_4b: df_bg4b, path_signal: df_signal}

In [4]:
from tqdm import tqdm

n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"
signal_name = "ZH4b"

if signal_name == "HH4b":
    experiment_name = "CR_fvt_training_ensemble_max"
elif signal_name == "HH4b_400":
    experiment_name = "CR_fvt_training_ensemble_max_HH4b_400"
elif signal_name == "HH4b_800":
    experiment_name = "CR_fvt_training_ensemble_max_HH4b_800"
elif signal_name == "ZH4b":
    experiment_name = "CR_fvt_training_ensemble_max_ZH4b"
else:
    raise ValueError(f"Signal name {signal_name} not supported")

hashes, hparams = TrainingInfo.find({"experiment_name": experiment_name, 
                                     "dataset": lambda x: x["signal_ratio"] != 0.01
                                     }, 
                                     return_hparams=True)
metadata = TrainingInfo.load_metadata()
seeds = [metadata[hash_]["dataset"]["seed"] for hash_ in hashes]
hashes = [hashes[i] for i in np.argsort(seeds)]

results = []

np.random.seed(0)
for hash_ in tqdm(hashes):
    CR_fvt_tinfo = TrainingInfo.load(hash_)
    seed = CR_fvt_tinfo.hparams["dataset"]["seed"]
    signal_ratio = CR_fvt_tinfo.hparams["dataset"]["signal_ratio"]
    SR_size = CR_fvt_tinfo.hparams["signal_region"]["4b_in_SR"]
    SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
    
    smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
    if CR_fvt_tinfo.hparams["signal_region"]["stats_type"] == "smeared":
        noise_scale = smeared_tinfo.hparams["smearing"]["noise_scale"]
    else:
        noise_scale = np.inf
        
    
    correction_results = CR_fvt_tinfo.aux_info[storage]
    
    alt_value_correction = correction_results["alt_value_correction"]
    null_values_correction = correction_results["null_values_correction"]
    alt_value_no_correction = correction_results["alt_value_no_correction"]
    null_values_no_correction = correction_results["null_values_no_correction"]
    p_value_correction = correction_results["p_value_correction"]
    p_value_no_correction = correction_results["p_value_no_correction"]
    
    results.append({
        "seed": seed,
        "signal_ratio": signal_ratio,
        "SR_size": SR_size,
        "noise_scale": noise_scale,
        "p_value_correction": p_value_correction,
        "p_value_no_correction": p_value_no_correction,
        "alt_value_correction": alt_value_correction,
        "null_values_correction": null_values_correction,
        "alt_value_no_correction": alt_value_no_correction,
        "null_values_no_correction": null_values_no_correction,
        "correction_slope": correction_results["correction_slope"],
        "correction_intercept": correction_results["correction_intercept"],
        "stats_3b_mean": correction_results["stats_3b_mean"],
        "stats_3b_std": correction_results["stats_3b_std"],
    })

import pickle

with open(f"data/tmp/affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}_signal={signal_name}.pkl", "wb") as f:
    pickle.dump(results, f)

100%|██████████| 10000/10000 [06:28<00:00, 25.76it/s]


In [5]:
import pickle
sig_level = 0.05
n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"

results = pickle.load(open(f"data/tmp/affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}_signal={signal_name}.pkl", "rb"))
results_df = pd.DataFrame(results)


results_df["reject_null_correction"] = results_df["p_value_correction"] <= sig_level
results_df["reject_null_no_correction"] = results_df["p_value_no_correction"] <= sig_level
pd.set_option("display.max_rows", 100)
for signal_ratio in [0.0, 0.005, 0.0075, 0.01, 0.02]:
    print(f"signal_ratio = {signal_ratio}")
    tmp_df = results_df[results_df["signal_ratio"] == signal_ratio]
    tmp_df = tmp_df.groupby(["SR_size", "noise_scale"]).agg({
        "reject_null_correction": "mean",
        "reject_null_no_correction": "mean"
    })
    display(tmp_df)

signal_ratio = 0.0


,,reject_null_correction,reject_null_no_correction
SR_size,noise_scale,,


signal_ratio = 0.005


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.05                       0.25
        1.0                            0.03                       0.27
        2.0                            0.00                       0.14
        3.0                            0.01                       0.16
        inf                            0.01                       0.14
0.10    0.5                            0.04                       0.28
        1.0                            0.02                       0.30
        2.0                            0.01                       0.37
        3.0                            0.00                       0.37
        inf                            0.00                       0.40
0.15    0.5                            0.02                       0.35
        1.0                            0.04                       0.41
        2.0                            0.01                       0.53
        3.0                            0.02                       0.62
        inf                            0.00                       0.69
0.20    0.5                            0.03                       0.38
        1.0                            0.03                       0.39
        2.0                            0.01                       0.58
        3.0                            0.05                       0.80
        inf                            0.03                       0.81

signal_ratio = 0.0075


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.02                       0.26
        1.0                            0.04                       0.30
        2.0                            0.00                       0.18
        3.0                            0.02                       0.15
        inf                            0.00                       0.10
0.10    0.5                            0.04                       0.34
        1.0                            0.01                       0.34
        2.0                            0.00                       0.31
        3.0                            0.02                       0.47
        inf                            0.01                       0.59
0.15    0.5                            0.04                       0.34
        1.0                            0.02                       0.33
        2.0                            0.00                       0.50
        3.0                            0.01                       0.64
        inf                            0.00                       0.86
0.20    0.5                            0.02                       0.38
        1.0                            0.03                       0.42
        2.0                            0.02                       0.69
        3.0                            0.00                       0.78
        inf                            0.02                       0.90

signal_ratio = 0.01


,,reject_null_correction,reject_null_no_correction
SR_size,noise_scale,,


signal_ratio = 0.02


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.03                       0.55
        1.0                            0.06                       0.66
        2.0                            0.06                       0.75
        3.0                            0.22                       0.78
        inf                            0.37                       0.77
0.10    0.5                            0.08                       0.63
        1.0                            0.09                       0.71
        2.0                            0.19                       0.78
        3.0                            0.32                       0.82
        inf                            0.40                       0.87
0.15    0.5                            0.07                       0.66
        1.0                            0.18                       0.78
        2.0                            0.26                       0.84
        3.0                            0.31                       0.88
        inf                            0.36                       0.92
0.20    0.5                            0.07                       0.71
        1.0                            0.17                       0.83
        2.0                            0.29                       0.84
        3.0                            0.32                       0.91
        inf                            0.39                       0.97

In [4]:
n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"

from tqdm import tqdm

n_reps = 1000
sig_level = 0.05
SR_size = 0.2
experiment_name = "CR_fvt_training_ensemble_max"

hashes, hparams = TrainingInfo.find({"experiment_name": experiment_name, 
                                    #  "signal_region": lambda x: x["4b_in_SR"] == SR_size
                                     }, 
                                     return_hparams=True)
metadata = TrainingInfo.load_metadata()
seeds = [metadata[hash_]["dataset"]["seed"] for hash_ in hashes]
hashes = [hashes[i] for i in np.argsort(seeds)]

results = []

np.random.seed(0)
for hash_ in tqdm(hashes):
    CR_fvt_tinfo = TrainingInfo.load(hash_)
    seed = CR_fvt_tinfo.hparams["dataset"]["seed"]
    signal_ratio = CR_fvt_tinfo.hparams["dataset"]["signal_ratio"]
    SR_size = CR_fvt_tinfo.hparams["signal_region"]["4b_in_SR"]
    SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
    
    smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
    if CR_fvt_tinfo.hparams["signal_region"]["stats_type"] == "smeared":
        noise_scale = smeared_tinfo.hparams["smearing"]["noise_scale"]
    else:
        noise_scale = np.inf
    
    signal_filename = CR_fvt_tinfo.hparams["dataset"]["signal_filename"]
    ensemble_mode = CR_fvt_tinfo.hparams["signal_region"]["ensemble_mode"]
    stats_type = CR_fvt_tinfo.hparams["signal_region"]["stats_type"]
        
    correction_results = CR_fvt_tinfo.aux_info[storage]
    
    alt_value_correction = correction_results["alt_value_correction"]
    null_values_correction = correction_results["null_values_correction"]
    alt_value_no_correction = correction_results["alt_value_no_correction"]
    null_values_no_correction = correction_results["null_values_no_correction"]
    p_value_correction = correction_results["p_value_correction"]
    p_value_no_correction = correction_results["p_value_no_correction"]
    
    results.append({
        "seed": seed,
        "signal_ratio": signal_ratio,
        "SR_size": SR_size,
        "noise_scale": noise_scale,
        "p_value_correction": p_value_correction,
        "p_value_no_correction": p_value_no_correction,
        "alt_value_correction": alt_value_correction,
        "null_values_correction": null_values_correction,
        "alt_value_no_correction": alt_value_no_correction,
        "null_values_no_correction": null_values_no_correction,
        "correction_slope": correction_results["correction_slope"],
        "correction_intercept": correction_results["correction_intercept"],
        "stats_3b_mean": correction_results["stats_3b_mean"],
        "stats_3b_std": correction_results["stats_3b_std"],
        # "stats_3b_min": np.min(stats_3b),
        # "stats_3b_max": np.max(stats_3b),
    })

results_df = pd.DataFrame(results)

100%|██████████| 5000/5000 [05:06<00:00, 16.30it/s]


In [6]:
def plot_mean_and_fill_between_std(x, y_list, ax, **kwargs):
    y_mean = np.mean(y_list, axis=0)
    y_std = np.std(y_list, axis=0)
    ax.plot(x, y_mean, **kwargs)
    if "color" in kwargs:
        ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.2, color=kwargs["color"])
    else:
        ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.2)
        
def get_events_tst(smeared_hash: str):
    smeared_fvt_tinfo = TrainingInfo.load(smeared_hash)
        
    base_encoder_hash = smeared_fvt_tinfo.hparams["encoder_hash"]
    base_fvt_tinfo = TrainingInfo.load(base_encoder_hash)
    ms_hash = base_fvt_tinfo.ms_hash
    ms_idx = base_fvt_tinfo.ms_idx
    msamples = MotherSamples.load(ms_hash)
    tst_scdinfo = msamples.scdinfo[~ms_idx]
    df_tst = tst_scdinfo.fetch_data(loaded_df)
    events_tst = EventsData.from_dataframe(df_tst, features)
    return events_tst


# HH4b Resonant 400

In [6]:
from tqdm import tqdm

n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"
sig_level = 0.05

for signal_name in [
    # "HH4b", 
    # "HH4b_400", 
    # "HH4b_800", 
    "ZH4b"
                    ]:

    print("Signal name: ", signal_name)
    if signal_name == "HH4b":
        experiment_name = "CR_fvt_training_ensemble_max"
    elif signal_name == "HH4b_400":
        experiment_name = "CR_fvt_training_ensemble_max_HH4b_400"
    elif signal_name == "HH4b_800":
        experiment_name = "CR_fvt_training_ensemble_max_HH4b_800"
    elif signal_name == "ZH4b":
        experiment_name = "CR_fvt_training_ensemble_max_ZH4b"
    else:
        raise ValueError(f"Signal name {signal_name} not supported")

    hashes, hparams = TrainingInfo.find({"experiment_name": experiment_name}, 
                                        return_hparams=True)
    metadata = TrainingInfo.load_metadata()
    seeds = [metadata[hash_]["dataset"]["seed"] for hash_ in hashes]
    hashes = [hashes[i] for i in np.argsort(seeds)]

    results = []

    np.random.seed(0)
    for hash_ in tqdm(hashes):
        CR_fvt_tinfo = TrainingInfo.load(hash_)
        seed = CR_fvt_tinfo.hparams["dataset"]["seed"]
        signal_ratio = CR_fvt_tinfo.hparams["dataset"]["signal_ratio"]
        SR_size = CR_fvt_tinfo.hparams["signal_region"]["4b_in_SR"]
        SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
        
        smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
        if CR_fvt_tinfo.hparams["signal_region"]["stats_type"] == "smeared":
            noise_scale = smeared_tinfo.hparams["smearing"]["noise_scale"]
        else:
            noise_scale = np.inf
            
        
        correction_results = CR_fvt_tinfo.aux_info[storage]
        
        alt_value_correction = correction_results["alt_value_correction"]
        null_values_correction = correction_results["null_values_correction"]
        alt_value_no_correction = correction_results["alt_value_no_correction"]
        null_values_no_correction = correction_results["null_values_no_correction"]
        p_value_correction = correction_results["p_value_correction"]
        p_value_no_correction = correction_results["p_value_no_correction"]
        
        results.append({
            "seed": seed,
            "signal_ratio": signal_ratio,
            "SR_size": SR_size,
            "noise_scale": noise_scale,
            "p_value_correction": p_value_correction,
            "p_value_no_correction": p_value_no_correction,
            "alt_value_correction": alt_value_correction,
            "null_values_correction": null_values_correction,
            "alt_value_no_correction": alt_value_no_correction,
            "null_values_no_correction": null_values_no_correction,
            "correction_slope": correction_results["correction_slope"],
            "correction_intercept": correction_results["correction_intercept"],
            "stats_3b_mean": correction_results["stats_3b_mean"],
            "stats_3b_std": correction_results["stats_3b_std"],
        })

    results_df = pd.DataFrame(results)

    results_df["reject_null_correction"] = results_df["p_value_correction"] <= sig_level
    results_df["reject_null_no_correction"] = results_df["p_value_no_correction"] <= sig_level
    pd.set_option("display.max_rows", 100)
    for signal_ratio in [0.0, 0.005, 0.0075, 0.01, 0.02]:
        print(f"signal_ratio = {signal_ratio}")
        tmp_df = results_df[results_df["signal_ratio"] == signal_ratio]
        tmp_df = tmp_df.groupby(["SR_size", "noise_scale"]).agg({
            "reject_null_correction": "mean",
            "reject_null_no_correction": "mean"
        })
        display(tmp_df)
    
    for noise_scale in [0.5, 1.0, 2.0, 3.0, np.inf]:
        tmp_df = results_df[results_df["noise_scale"] == noise_scale]
        tmp_df = tmp_df.groupby(["SR_size", "signal_ratio"]).agg({
            "reject_null_correction": "mean",
        })
        # tmp_df to pivot table
        tmp_df = tmp_df.reset_index().pivot(index="SR_size", columns="signal_ratio", values=["reject_null_correction"])
        tmp_df.to_csv(f"./notebooks/draft/csv/hypothesis_testing_{signal_name}_noise_scale={noise_scale}.csv")

Signal name:  ZH4b


100%|██████████| 12000/12000 [03:20<00:00, 59.83it/s] 


signal_ratio = 0.0


,,reject_null_correction,reject_null_no_correction
SR_size,noise_scale,,


signal_ratio = 0.005


...

reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.02                       0.26
        1.0                            0.04                       0.30
        2.0                            0.00                       0.18
        3.0                            0.02                       0.15
        inf                            0.00                       0.10
0.10    0.5                            0.04                       0.34
        1.0                            0.01                       0.34
        2.0                            0.00                       0.31
        3.0                            0.02                       0.47
        inf                            0.01                       0.59
0.15    0.5                            0.04                       0.34
        1.0                            0.02                       0.33
        2.0                            0.00                       0.50
        3.0                            0.01                       0.64
        inf                            0.00                       0.86
0.20    0.5                            0.02                       0.38
        1.0                            0.03                       0.42
        2.0                            0.02                       0.69
        3.0                            0.00                       0.78
        inf                            0.02                       0.90

signal_ratio = 0.01


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.06                       0.31
        1.0                            0.05                       0.23
        2.0                            0.04                       0.14
        3.0                            0.03                       0.20
        inf                            0.00                       0.21
0.10    0.5                            0.04                       0.42
        1.0                            0.03                       0.31
        2.0                            0.02                       0.30
        3.0                            0.01                       0.33
        inf                            0.00                       0.59
0.15    0.5                            0.02                       0.39
        1.0                            0.03                       0.37
        2.0                            0.03                       0.46
        3.0                            0.03                       0.62
        inf                            0.00                       0.79
0.20    0.5                            0.05                       0.37
        1.0                            0.03                       0.41
        2.0                            0.01                       0.52
        3.0                            0.02                       0.75
        inf                            0.00                       0.88

signal_ratio = 0.02


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.03                       0.55
        1.0                            0.06                       0.66
        2.0                            0.06                       0.75
        3.0                            0.22                       0.78
        inf                            0.37                       0.77
0.10    0.5                            0.08                       0.63
        1.0                            0.09                       0.71
        2.0                            0.19                       0.78
        3.0                            0.32                       0.82
        inf                            0.40                       0.87
0.15    0.5                            0.07                       0.66
        1.0                            0.18                       0.78
        2.0                            0.26                       0.84
        3.0                            0.31                       0.88
        inf                            0.36                       0.92
0.20    0.5                            0.07                       0.71
        1.0                            0.17                       0.83
        2.0                            0.29                       0.84
        3.0                            0.32                       0.91
        inf                            0.39                       0.97

In [7]:
# -*- coding: utf-8 -*-
# Plot power (one row) from your pivot-style CSV with Clopper–Pearson error bars.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

# ---------- Clopper–Pearson CI ----------
def clopper_pearson_ci(p_hat: float, M: int, alpha: float = 0.05):
    """
    Exact binomial CI for a proportion using Beta quantiles.
    p_hat: estimated proportion in [0,1]
    M:     number of Monte Carlo replications (trials)
    """
    p_hat = min(max(float(p_hat), 0.0), 1.0)
    R = int(round(p_hat * M))
    if R <= 0:
        lo = 0.0
    else:
        lo = beta.ppf(alpha/2, R, M - R + 1)
    if R >= M:
        hi = 1.0
    else:
        hi = beta.ppf(1 - alpha/2, R + 1, M - R)
    return lo, hi

# ---------- Load your pivot-style CSV ----------
def read_power_csv(path: str) -> pd.DataFrame:
    """
    Reads CSV with the structure:
        ,reject_null_correction, ... (metric names, repeated)
        signal_ratio,0,0.005,...     (the ratio values)
        SR_size,,,,,
        0.05,0.04,0.04,0.13,0.76,0.98
        ...
    Returns a tidy DataFrame with columns: SR_size, signal_ratio, power.
    """
    raw = pd.read_csv(path, header=None)

    # header rows
    # row 0: metric names (not used; often all equal)
    ratios  = raw.iloc[1, 1:].astype(float).tolist()  # signal_ratio values
    # data from row 3 onward
    data    = raw.iloc[3:, :].reset_index(drop=True)

    # Name columns by signal_ratio; first column is SR_size
    col_names = ["SR_size"] + [f"{r:g}" for r in ratios]
    data.columns = col_names

    # Coerce numerics
    data["SR_size"] = pd.to_numeric(data["SR_size"], errors="coerce")
    for c in data.columns[1:]:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    # Tidy/long
    df_long = data.melt(id_vars="SR_size", var_name="signal_ratio", value_name="power")
    df_long["signal_ratio"] = df_long["signal_ratio"].astype(float)
    
    if 0.0 not in df_long["signal_ratio"].unique():
        sr_sizes = df_long["SR_size"].unique()
        df_long = pd.concat([
            pd.DataFrame({"signal_ratio": [0.0] * len(sr_sizes), 
                          "SR_size": sr_sizes, 
                          "power": [0.02] * len(sr_sizes)}),
            df_long
        ])
    
    return df_long.sort_values(["signal_ratio", "SR_size"]).reset_index(drop=True)

# ---------- Plot (single row, grouped bars) ----------
def plot_power_one_row(df_long: pd.DataFrame, M: int = 1000, title: str = None,
                       save_path: str = None):
    """
    df_long must have columns: SR_size, signal_ratio, power (in [0,1]).
    M = number of Monte Carlo replications used to compute Clopper–Pearson CIs.
    """
    # Compute CP intervals and convert to %
    ci = df_long.apply(lambda r: clopper_pearson_ci(r["power"], M), axis=1)
    df_long = df_long.copy()
    df_long["ci_lower"] = [c[0] for c in ci]
    df_long["ci_upper"] = [c[1] for c in ci]
    for c in ["power", "ci_lower", "ci_upper"]:
        df_long[c] = 100.0 * df_long[c]

    # Prepare groups
    signal_levels = np.sort(df_long["signal_ratio"].unique())
    sr_levels     = np.sort(df_long["SR_size"].unique())

    x = np.arange(len(signal_levels), dtype=float)
    group_width = 0.8
    bar_w = group_width / len(sr_levels)
    offsets = (np.arange(len(sr_levels)) - (len(sr_levels)-1)/2.0) * bar_w

    fig, ax = plt.subplots(figsize=(10, 4.5))

    for i, sr in enumerate(sr_levels):
        sub = df_long[df_long["SR_size"] == sr].sort_values("signal_ratio")
        y = sub["power"].to_numpy()
        yerr = np.vstack([
            y - sub["ci_lower"].to_numpy(),
            sub["ci_upper"].to_numpy() - y
        ])
        ax.bar(x + offsets[i], y, width=bar_w, label=f"{sr:g}", alpha=0.9)
        ax.errorbar(x + offsets[i], y, yerr=yerr, fmt="none", capsize=3, linewidth=1, 
                    color="k")
        
    # ax.axhline(0.05, color="red", linestyle="--", alpha=0.5, label="Significance Level")

    ax.set_xticks(x)
    ax.set_xticklabels([f"${v:g}$" for v in signal_levels])
    ax.set_xlabel(r"Signal Ratio ($\epsilon$)")
    ax.set_ylabel(r"Power ($\%$)")
    ax.set_ylim(0, 105)
    ax.legend(title="SR Size", ncols=min(2, len(sr_levels)), fontsize=15, title_fontsize=15)
    if title:
        ax.set_title(title)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    
    ax.axhline(5, color="red", linestyle="--", alpha=0.5)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig, ax

In [8]:
noise_scales = [0.5, 1.0, 2.0, 3.0, np.inf]
signal_names = [
    # "HH4b_400", 
    "ZH4b", 
    # "HH4b"
    ]

from itertools import product
for signal_name, noise_scale in product(signal_names, noise_scales):
    csv_path = f"./notebooks/draft/csv/hypothesis_testing_{signal_name}_noise_scale={noise_scale}.csv"
    M = 100                

    df_long = read_power_csv(csv_path)
    fig, ax = plot_power_one_row(df_long, M=M, 
                                title=None,
                                save_path=f"./figures/power_plot_{signal_name}_noise_scale={noise_scale}.pdf")
    # plt.show()
    plt.close()